# 09 — Recommender-System Metrics

Covers the full recsys metrics suite — from data preparation to monitoring with the Runner.

- `RecSysSchema` — column role declaration
- `interactions_from_matrix` — wide matrix → long interactions table
- **Ranking accuracy**: `precision_at_k`, `recall_at_k`, `fbeta_at_k`, `hit_rate`, `map_at_k`, `ndcg_at_k`, `mrr_at_k`
- **Beyond accuracy**: `diversity`, `novelty`, `popularity_bias`, `personalization`, `item_bias`, `user_bias`
- Running all recsys metrics via `MonitoringPlan` + `Runner`

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Synthetic interaction data

We simulate a small e-commerce recommendation scenario:
- 50 users, 20 items
- Each user has a ground-truth relevant set (the items they actually bought)
- A recommender produces a score for every (user, item) pair
- Items also carry two content features (used by the Diversity metric)

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

N_USERS = 50
N_ITEMS = 20

users = [f"user_{i:03d}" for i in range(N_USERS)]
items = [f"item_{j:03d}" for j in range(N_ITEMS)]

# Item content features (2-D; used by Diversity)
item_features = rng.standard_normal((N_ITEMS, 2))

# Build interactions table — one row per (user, item)
rows = []
for u_idx, user in enumerate(users):
    # Each user finds ~4 items relevant
    relevant_items = rng.choice(N_ITEMS, size=4, replace=False)
    for j, item in enumerate(items):
        relevance = 1 if j in relevant_items else 0
        # Score correlates with relevance + noise
        score = relevance * 0.6 + rng.uniform(0, 0.4)
        rows.append({
            "user_id": user,
            "item_id": item,
            "relevance": relevance,
            "score": round(score, 4),
            "feat_0": round(item_features[j, 0], 4),
            "feat_1": round(item_features[j, 1], 4),
        })

df = pd.DataFrame(rows)
print(df.shape)
df.head(6)

Also create a **training interaction log** (used by `novelty` and `popularity_bias`  
to estimate item popularity from historical data).

In [ ]:
# Training log: popular items (first 5) appear far more often
train_items = rng.choice(
    items,
    size=2000,
    p=(_p := np.array([0.15, 0.12, 0.10, 0.08, 0.07] + [0.03] * 15)) / _p.sum(),
)
df_train = pd.DataFrame({
    "user_id": [f"user_{rng.integers(200):03d}" for _ in range(2000)],
    "item_id": train_items,
    "relevance": np.ones(2000, dtype=int),
})
print("Training interactions:", df_train.shape)
df_train["item_id"].value_counts().head(8)

## 2. RecSysSchema

`RecSysSchema` maps logical column roles to the physical column names in your DataFrame.

In [ ]:
from ayn_ml.core.schema import RecSysSchema

schema = RecSysSchema(
    user_id_col="user_id",
    item_id_col="item_id",
    relevance_col="relevance",
    score_col="score",              # predicted scores
    recommendations_type="score",   # "score" (higher = better) | "rank" (lower = better)
)
print(schema)
print("Column names declared:", schema.column_names)

## 3. interactions_from_matrix

If your data lives in a **user×item matrix**, convert it to the long format first.

In [ ]:
from ayn_ml.metrics.recsys_utils import interactions_from_matrix

# Build 5×5 example matrices
truth_matrix = pd.DataFrame(
    rng.integers(0, 2, size=(5, 5)),
    index=[f"u{i}" for i in range(5)],
    columns=[f"i{j}" for j in range(5)],
)
pred_matrix = pd.DataFrame(
    rng.uniform(0, 1, size=(5, 5)).round(3),
    index=truth_matrix.index,
    columns=truth_matrix.columns,
)

# Convert to long format
long_df = interactions_from_matrix(truth_matrix, pred_matrix)
print(long_df.shape)  # 25 rows (5 users × 5 items)
long_df.head(6)

## 4. Ranking accuracy metrics

All metrics read `k` (cutoff) and `relevance_threshold` from `MetricSpec.params`.

In [ ]:
from ayn_ml.core.spec import MetricSpec
from ayn_ml.metrics.recsys import (
    FBetaAtKMetric,
    HitRateMetric,
    MAPAtKMetric,
    MRRAtKMetric,
    NDCGAtKMetric,
    PrecisionAtKMetric,
    RecallAtKMetric,
)

K = 5
ranking_metrics = [
    ("precision_at_k", PrecisionAtKMetric()),
    ("recall_at_k",    RecallAtKMetric()),
    ("f1_at_k",        FBetaAtKMetric()),      # beta=1 by default
    ("hit_rate",       HitRateMetric()),
    ("map_at_k",       MAPAtKMetric()),
    ("ndcg_at_k",      NDCGAtKMetric()),
    ("mrr_at_k",       MRRAtKMetric()),
]

for name, metric in ranking_metrics:
    spec = MetricSpec(name=name, params={"k": K})
    result = metric.compute(df, None, schema, spec)
    print(f"{name:<18} = {result.value:.4f}")

### Effect of k on precision vs recall trade-off

In [ ]:
ks = [1, 3, 5, 10, 15]
print(f"{'k':>4}  {'precision':>10}  {'recall':>10}")
for k in ks:
    p = PrecisionAtKMetric().compute(df, None, schema, MetricSpec(name="p", params={"k": k})).value
    r = RecallAtKMetric().compute(df, None, schema, MetricSpec(name="r", params={"k": k})).value
    print(f"{k:>4}  {p:>10.4f}  {r:>10.4f}")

### Graded relevance with NDCG

In [ ]:
# Replace binary relevance with graded scores (1–5)
df_graded = df.copy()
df_graded["relevance"] = df_graded["relevance"] * rng.integers(1, 6, size=len(df_graded))

ndcg_binary = NDCGAtKMetric().compute(df,        None, schema, MetricSpec(name="n", params={"k": 5})).value
ndcg_graded = NDCGAtKMetric().compute(df_graded, None, schema, MetricSpec(name="n", params={"k": 5})).value

print(f"NDCG@5 (binary relevance): {ndcg_binary:.4f}")
print(f"NDCG@5 (graded relevance): {ndcg_graded:.4f}")

## 5. Beyond-accuracy metrics

In [ ]:
from ayn_ml.metrics.recsys import (
    DiversityMetric,
    ItemBiasMetric,
    NoveltyMetric,
    PersonalizationMetric,
    PopularityBiasMetric,
    UserBiasMetric,
)

K = 5

# Diversity — needs item feature columns
div = DiversityMetric().compute(
    df, None, schema,
    MetricSpec(name="diversity", params={"k": K, "item_features": ["feat_0", "feat_1"]})
).value

# Novelty & PopularityBias — need training data as reference
nov = NoveltyMetric().compute(
    df, df_train, schema, MetricSpec(name="novelty", params={"k": K})
).value
pop = PopularityBiasMetric().compute(
    df, df_train, schema, MetricSpec(name="popularity_bias", params={"k": K})
).value

# Personalization, ItemBias, UserBias — no reference needed
pers = PersonalizationMetric().compute(
    df, None, schema, MetricSpec(name="personalization", params={"k": K})
).value
ibias = ItemBiasMetric().compute(
    df, None, schema, MetricSpec(name="item_bias", params={"k": K})
).value
ubias = UserBiasMetric().compute(
    df, None, schema, MetricSpec(name="user_bias", params={"k": K})
).value

print(f"diversity        = {div:.4f}   (0=identical items, 1=orthogonal)")
print(f"novelty          = {nov:.4f}   (higher = less popular items)")
print(f"popularity_bias  = {pop:.4f}   (higher = more popular items)")
print(f"personalization  = {pers:.4f}   (0=same lists, 1=fully disjoint)")
print(f"item_bias (Gini) = {ibias:.4f}   (0=equal spread, →1=concentrated)")
print(f"user_bias (Gini) = {ubias:.4f}   (0=all users get k items)")

## 6. Full monitoring run with Runner

Combine all metrics into a `MonitoringPlan` and run them in one pass.

In [ ]:
from ayn_ml.core.spec import MonitoringPlan
from ayn_ml.runner import Runner

plan = MonitoringPlan(
    name="recsys_monitoring",
    model_id="recommender_v1",
    model_version="1.0.0",
    data_schema=schema,
    metrics=[
        MetricSpec(name="precision_at_k",   params={"k": 5}, threshold=0.3, upper_bound=False),
        MetricSpec(name="recall_at_k",       params={"k": 5}),
        MetricSpec(name="ndcg_at_k",         params={"k": 5}, threshold=0.4, upper_bound=False),
        MetricSpec(name="map_at_k",          params={"k": 5}),
        MetricSpec(name="mrr_at_k",          params={"k": 5}),
        MetricSpec(name="hit_rate",          params={"k": 5}),
        MetricSpec(name="diversity",         params={"k": 5, "item_features": ["feat_0", "feat_1"]}),
        MetricSpec(name="novelty",           params={"k": 5}),
        MetricSpec(name="popularity_bias",   params={"k": 5}, threshold=0.1, upper_bound=True),
        MetricSpec(name="personalization",   params={"k": 5}),
        MetricSpec(name="item_bias",         params={"k": 5}, threshold=0.5, upper_bound=True),
        MetricSpec(name="user_bias",         params={"k": 5}),
    ],
)

runner = Runner()
report = runner.run(plan, current=df, reference=df_train)

print(f"Metrics computed : {len(report.results)}")
print(f"Errors           : {len(report.errors)}")
for r in report.results:
    status = ("pass" if r.status else "fail") if r.status is not None else "—"
    print(f"  {r.spec.name:<20} = {r.value:.4f}   [{status}]")

## 7. Rank mode

If your model outputs explicit rank positions instead of scores, set `recommendations_type="rank"`.
The lower the rank, the better (rank 1 = best recommendation).

In [ ]:
# Convert scores to ranks (per user, rank 1 = highest score)
df_ranked = df.copy()
df_ranked["rank"] = (
    df_ranked.groupby("user_id")["score"]
    .rank(ascending=False, method="first")
    .astype(int)
)

schema_rank = RecSysSchema(
    user_id_col="user_id",
    item_id_col="item_id",
    relevance_col="relevance",
    score_col="rank",
    recommendations_type="rank",
)

p_score = PrecisionAtKMetric().compute(df,        None, schema,      MetricSpec(name="p", params={"k": 5})).value
p_rank  = PrecisionAtKMetric().compute(df_ranked, None, schema_rank, MetricSpec(name="p", params={"k": 5})).value

print(f"Precision@5 (score mode) = {p_score:.4f}")
print(f"Precision@5 (rank mode)  = {p_rank:.4f}")
assert abs(p_score - p_rank) < 1e-4, "Should be identical — same ordering"